## SCOPE-3 SUPPLIER EMISSIONS PREDICTION MODEL

Trains and evaluates multiple regression models to predict supplier-level Scope-3 emissions.
**Input:** `data/supplier_summary_with_priority.csv` (output of scope3_pipeline.ipynb)
**Output:** `data/supplier_predictions.csv`, `data/model_comparison.csv`, `data/feature_importance.csv`


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

import plotly.graph_objects as go
import plotly.express as px

## 1. Load Pipeline Output

In [ ]:
# Run scope3_pipeline.ipynb first to generate this file
df = pd.read_csv('data/supplier_summary_with_priority.csv')
print(f'Loaded {len(df)} supplier-period records')
df.head()

## 2. Feature Engineering

In [ ]:
# Log transform to handle skewed distributions
df['log_spend']     = np.log1p(df['Total_Spend_USD'])
df['log_emissions'] = np.log1p(df['Total_Emissions_tCO2e'])

# Derived feature: spend efficiency
df['spend_per_po'] = df['Total_Spend_USD'] / df['PO_Lines'].replace(0, 1)

# One-hot encode Period (better than LabelEncoder for regression)
df = pd.get_dummies(df, columns=['Period'], drop_first=True)

feature_cols = ['log_spend', 'PO_Lines', 'spend_per_po'] + [col for col in df.columns if col.startswith('Period_')]
target = 'log_emissions'

df = df.dropna(subset=[target])
print('Features:', feature_cols)
print('Records after dropna:', len(df))

## 3. Train / Validation Split

In [ ]:
# Business-based split: top spenders as validation set (most critical to get right)
df_sorted = df.sort_values('Total_Spend_USD', ascending=False).reset_index(drop=True)
val_size  = max(int(len(df_sorted) * 0.15), 5)

val_df   = df_sorted.iloc[:val_size]
train_df = df_sorted.iloc[val_size:]

X_train, y_train = train_df[feature_cols], train_df[target]
X_val,   y_val   = val_df[feature_cols],   val_df[target]

print(f'Train: {len(X_train)}, Validation: {len(X_val)}')

## 4. Model Definitions

In [ ]:
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ]),
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0))
    ]),
    'Random Forest':      RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, random_state=42),
}

## 5. Train & Evaluate

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    results[name] = {
        'model': model, 'preds': preds,
        'MAE':  mean_absolute_error(y_val, preds),
        'R2':   r2_score(y_val, preds),
        'RMSE': np.sqrt(mean_squared_error(y_val, preds)),
        'MAPE': mape(y_val, preds)
    }

comp = pd.DataFrame([
    {'Model': k, 'MAE': round(v['MAE'],4), 'R2': round(v['R2'],4),
     'RMSE': round(v['RMSE'],4), 'MAPE (%)': round(v['MAPE'],2)}
    for k, v in results.items()
])
print('\nMODEL COMPARISON:')
print(comp)
comp.to_csv('data/model_comparison.csv', index=False)

## 6. Select Best Model & Generate Predictions

In [ ]:
best_name  = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f'\nBest Model: {best_name}')

df['pred_log']           = best_model.predict(df[feature_cols])
df['Predicted_Emissions'] = np.expm1(df['pred_log']).clip(lower=0)

df.to_csv('data/supplier_predictions.csv', index=False)
print('Saved supplier_predictions.csv')

## 7. Feature Importance

In [ ]:
model_core = best_model.named_steps['model'] if hasattr(best_model, 'named_steps') else best_model

if hasattr(model_core, 'feature_importances_'):
    fi = pd.Series(model_core.feature_importances_, index=feature_cols)
else:
    fi = pd.Series(np.abs(model_core.coef_), index=feature_cols)

fi = fi.sort_values(ascending=False)
fi.to_csv('data/feature_importance.csv')
print('Saved feature_importance.csv')
print(fi)

## 8. Visualisation — Actual vs Predicted Emissions

In [ ]:
fig = px.bar(
    comp, x='Model', y='R2',
    title='Model Comparison — R² Score',
    color='R2', color_continuous_scale='Teal'
)
fig.show()

# Actual vs Predicted scatter (best model)
preds_all = best_model.predict(df[feature_cols])
actual    = np.expm1(df[target])
predicted = np.expm1(preds_all).clip(min=0)

fig2 = px.scatter(
    x=actual, y=predicted,
    labels={'x': 'Actual Emissions (tCO2e)', 'y': 'Predicted Emissions (tCO2e)'},
    title=f'Actual vs Predicted Emissions — {best_name}'
)
fig2.add_shape(type='line', x0=actual.min(), y0=actual.min(),
               x1=actual.max(), y1=actual.max(),
               line=dict(color='red', dash='dash'))
fig2.show()